## Установка нужных версий библиотек

In [1]:
!pip install scikit-learn pandas matplotlib numpy 

In [2]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
import pandas as pd
import matplotlib.pyplot as plt

### 1. Получение данных load_breat_cancer

Будем работать с набором данных для задачи регрессии `load_diabetes`, который можно получить из стандартных датасетов в `sklearn'e`.

После `load_diabetes()` возвращается словарь с данными (`data`), целевой переменной (`target`), названиями характеристик в данных (`feature_names`) и описанием данных (`DESCR`).

In [3]:
from sklearn.datasets import load_diabetes

data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]]),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142., 155., 225.,  59

In [4]:
X = data.data
features = data.feature_names
y = data.target

Из признаков (характеристик данных) и целевой переменной сформируем датафрейм, в качестве названий колонок возьмем названия признаков.

In [5]:
df = pd.DataFrame(X, columns=features)
df['target'] = y

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


Разобьем выборку на две: обучающую и тестовую.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df['target'],
    test_size=0.2,
    shuffle=True,
    random_state=3
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((353, 10), (353,), (89, 10), (89,))

### 1.1. Обучение дерева решений

1. Инициализируйте дерево решений для задачи регрессии
2. Обучите его на обучающей выборке

In [7]:
# Ваш код здесь
from sklearn.tree import DecisionTreeRegressor


tree = DecisionTreeRegressor(random_state=1)
tree.fit(X_train, y_train)

DecisionTreeRegressor(random_state=1)

### 1.2. Получение метрик качества
Узнаем, насколько дерево решений обучилось хорошо, для этого
1. Сделайте предсказания моделью для обучающей выборки
2. Сделайте предсказания моделью для тестовой выборки
3. Посчитайте метрику качества средне-квадратичная ошибка
4. Посчитайте метрику качества коэффициент детерминации

In [ ]:
# Ваш код здесь
from sklearn.metrics import mean_squared_error, r2_score

pred_train = tree.predict(X_train)
pred_test = tree.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)

print(f'MSE на обучении {mse_train:.2f}, r2_score {r2_score(y_train, pred_train):.3f}')
print(f'MSE на тесте {mse_test:.2f}, r2_score {r2_score(y_test, pred_test):.3f}')

MSE на обучении 0.00, r2_score 1.000
MSE на тесте 5897.13, r2_score -0.089


Сделайте вывод, насколько хорошо обучилась модель

Модель вышла переобученной, метрика на обучении идеальная, а вот на тесте не совсем.

Переобучение (вопреки утверджениям мелких) является не обучением модели заново, а зазубриванием модели тренировочных данных. Из второй части лабы мы знаем, что дерево строится до тех пор, пока все элементы в его листах не будут одного класса (или пока разбиение не будет невозможно, например, когда признаки у sample'ов одинаковые, но классы разные). Поскольку в регресси у нас (обычно) все целевые значения уникальные, дерево разбивается до тех пор, пока у него в листе не останется только одно значение.

Почему это плохо? У тестовых данных будут цены другие. Своим разбиением мы будем уходить к "идеальному" с точки зрения обучающих данных предсказаниям, но проигрывать полностью на новых данных, которых модель не видела. Поэтому лучше дерево не будет педантом, и будет разбивать не до одного, но до какой-то группы примеров, а потом брать среднее. В конце концов с KNN мы же тоже брали среднее по K-соседям, а не только самого ближайшего.

### 1.3. Изменение метрики

Попробуйте поизменять известные параметры для того, чтобы метрика стала лучше.

## Bruh

Это не метрики, и даже не параметры, а гиперпараметры.

- Метрики - функции для измерения качества работы модели (accuracy, MSE)
- Параметры модели - обученные параметры, которые используются для предсказания (веса у Линейной Регрессии, критерии разбиения у Дерева Решений)
- Гиперпараметры модели - задаваемые "характеристики" модели (количество соседей у KNN, или вот у дерева дальше)

И лучше подбирать гиперпараметры на валидационной выборке (третий тип выборки, по которому и подбирают гиперпараметры и отслеживают переобучение). Можно сделать сплит тренировочных данных, и 20% отдать на валидацию. Можно и на тестовой, никто не запрещает, просто результат будет стремиться переобучиться не на обучающих, а на тестовых данных. Сделать это  модель не сможет из-за того, что тестовых он не видит, но всё равно ориентировка на тест полу-bruh.

Ну и подбирать не ручками, а специальными функциями, типа Байесовских методов в Optuna или перебором с GridSearch из sklearn. Примеры есть в папке в этом же предмете в папке labs в 5 лабе с optuna или в AIS в 3 практике с GridSearch (в другом репозитории dl_examples тоже в одной из DLS домашек 1 семестра, но я не помню, где именно. Можете Ctrl+Shift+F на гридсёрч).

In [ ]:
# Ваш код здесь

tree = DecisionTreeRegressor(random_state=1,
                             max_depth=3,
                             min_samples_leaf=25,
                             max_leaf_nodes=5)
tree.fit(X_train, y_train)

pred_train = tree.predict(X_train)
pred_test = tree.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)

print(f'MSE на обучении {mse_train:.2f}, r2_score {r2_score(y_train, pred_train):.3f}')
print(f'MSE на тесте {mse_test:.2f}, r2_score {r2_score(y_test, pred_test):.3f}')

MSE на обучении 3212.06, r2_score 0.469
MSE на тесте 3239.92, r2_score 0.402


Более менее не переобученное дерево решений получилось с такими признаками.
Метрика r2_score всё равно не очень большая на тесте, но из модели как будто мы выжали все соки.

### 2. Получение данных make_regression

Для второго примера возьмем самодельный набор данных для задачи регрессии `make_regression`, который можно получить из стандартных датасетов в `sklearn'e`.

Сгенерируем себе 100к объектов, которые описываются 20 признаками, из них 12 будут дейтсвительно полезными.

In [8]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=100_000, n_features=20, n_informative=12, random_state=10)

In [ ]:
X.shape, y.shape

((100000, 20), (100000,))

Разобьем выборку на две: обучающую и тестовую.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=3
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((80000, 20), (80000,), (20000, 20), (20000,))

### 2.1. Обучение дерева решений

1. Инициализируйте дерево решений для задачи регрессии
2. Обучите его на обучающей выборке

In [ ]:
# Ваш код здесь
from sklearn.tree import DecisionTreeRegressor


tree = DecisionTreeRegressor(random_state=1)
tree.fit(X_train, y_train)

DecisionTreeRegressor(ccp_alpha=0.0, criterion='mse', max_depth=None,
                      max_features=None, max_leaf_nodes=None,
                      min_impurity_decrease=0.0, min_impurity_split=None,
                      min_samples_leaf=1, min_samples_split=2,
                      min_weight_fraction_leaf=0.0, presort='deprecated',
                      random_state=1, splitter='best')

### 2.2. Получение метрик качества
Узнаем, насколько дерево решений обучилось хорошо, для этого
1. Сделайте предсказания моделью для обучающей выборки
2. Сделайте предсказания моделью для тестовой выборки
3. Посчитайте метрику качества средне-квадратичная ошибка
4. Посчитайте метрику качества коэффициент детерминации

In [ ]:
# Ваш код здесь
from sklearn.metrics import mean_squared_error, r2_score

pred_train = tree.predict(X_train)
pred_test = tree.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)

print(f'MSE на обучении {mse_train:.2f}, r2_score {r2_score(y_train, pred_train):.3f}')
print(f'MSE на тесте {mse_test:.2f}, r2_score {r2_score(y_test, pred_test):.3f}')

MSE на обучении 0.00, r2_score 1.000
MSE на тесте 16658.27, r2_score 0.691


Сделайте вывод, насколько хорошо обучилась модель

In [ ]:
# Ваш вывод здесь

И снова дерево решений переобучилось.

невероятно!

### 2.3. Изменение метрики

Попробуйте поизменять известные параметры для того, чтобы метрика стала лучше.

## Bruh

Почему Bruh - вернитесь к аналогичному пункту первой половины

In [ ]:
# Ваш код здесь

tree = DecisionTreeRegressor(random_state=1,
                             max_depth=12,
                             min_samples_leaf=8,
                             max_leaf_nodes=3000)
tree.fit(X_train, y_train)

pred_train = tree.predict(X_train)
pred_test = tree.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)

print(f'MSE на обучении {mse_train:.2f}, r2_score {r2_score(y_train, pred_train):.3f}')
print(f'MSE на тесте {mse_test:.2f}, r2_score {r2_score(y_test, pred_test):.3f}')

MSE на обучении 7987.97, r2_score 0.852
MSE на тесте 14964.03, r2_score 0.723


И снова более менее не переобученное дерево решений с довольно высокой метрикой на тесте.

Сегодня на практическом занятии:
1. Обучили дерево решений без критериев останова
2. Изменяли знакомые три критерия останова, чтобы снизить переобучение
3. Обучили две модели